[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/094.%20The%20Multi-Echelon%20Inventory%20Optimization%20Problem/P94-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 94. The Multi-Echelon Inventory Optimization Problem

## Tier 8: A Value-Aligned Framework for Equitable Service Levels

### Purpose and scope

This notebook demonstrates how stakeholder priorities, vulnerability, service targets, and equity penalties can be represented in an inventory-planning example. It compares a priority-adjusted inventory rule with a fixed cost-only benchmark rule, then reports service, equity, stakeholder utility, and compliance measures.

The priority-adjusted rule is an illustrative heuristic, not a mathematical optimizer: it does not search over inventory decisions or guarantee an optimal solution. In particular, the priority and vulnerability adjustments determine inventory directly; the equity penalty is calculated afterward as an evaluation score. The benchmark is a fixed inventory rule included to give the reported metrics context; it does not optimize inventory with respect to costs and is not an alternative optimization method.

The example uses synthetic demand and cost data for a single planning period. Demand and inventory are measured in inventory units per period; `base_costs` are holding-cost dollars per inventory unit. Service level is a simplified location-level proxy, not a SKU-specific fill rate. Although the policy records distinguish essential medicines, baby formula, and basic food, this example has no SKU-level demand data, so those policies are evaluated against the same location-level service proxy. Penalty weights are dollar-equivalent modeling parameters, not calibrated financial estimates.

The model does not represent supplier lead times, replenishment decisions, transportation, capacity, stochastic demand distributions, or flows between echelons. It also does not implement constitutional AI or a formal multi-objective optimizer. Its principles and reports provide a transparent illustration of how ethical criteria can be recorded and assessed; they do not establish that a real allocation is fair or compliant.

### Modeling assumptions

- Higher priority and vulnerability scores lead to larger inventory targets through fixed multiplicative adjustments.
- Location service is estimated from inventory relative to demand and is capped at 99%.
- Each listed location belongs to one stakeholder group. Group service is the unweighted mean of the service levels at its locations.
- Equity penalties add the weighted service shortfall for each applicable constraint and location. Constraints with overlapping policy labels can therefore contribute separate penalties to the same location-level proxy.
- The equity score measures similarity in average service across represented stakeholder groups; it does not measure whether service is high enough or whether populations are treated fairly in every relevant respect.

### Stakeholders and principles

The example represents urban high-income and middle-income areas, rural communities, elderly care facilities, and schools and pediatric care. Each group has locations, a priority score, a vulnerability score, a population served, and stated service requirements. The requirements are recorded for context; the inventory heuristic does not directly enforce them.

The registered principles are service-level equity, priority for vulnerable populations, transparency, sustainability, and stakeholder accountability. The implemented inventory rule uses priority and vulnerability scores. Service shortfalls contribute to the reported equity penalty, while the other principles are represented in the reporting structure rather than as independently optimized objectives.

### Decision rule and evaluation equations

For a location assigned to stakeholder group \(g\), the heuristic starts with \(1.5d_i\) units of inventory and applies the group's vulnerability and priority multipliers. The location-level service proxy is then calculated from the resulting inventory. These equations describe the rule used in the code; they do not define an optimization problem with a search over feasible inventory plans.

$$
I_i = 1.5d_i\,v_g\,p_g,
\qquad
s_i = \min\!\left(0.99,\frac{I_i}{2d_i}\right),
$$

where \(i\) denotes a location, \(d_i>0\) is its demand, \(I_i\) is its inventory, and \(s_i\) is its estimated service level. The vulnerability multiplier is \(v_g=1.3\) when the group's vulnerability score is greater than \(0.7\), and \(1\) otherwise. The priority multiplier is \(p_g=1.2\) when the group's priority is greater than \(0.8\), and \(1\) otherwise. In the code, these calculations appear in `optimize_ethical_inventory`.

The fixed cost-only benchmark uses a separate rule: it assigns inventory equal to 1.2 times demand and estimates service using a denominator of 1.5 times demand, capped at 90%. The cost inputs are used to calculate the resulting holding cost; they do not determine the benchmark's inventory quantities.

$$
I_i^{B}=1.2d_i,
\qquad
s_i^{B}=\min\!\left(0.90,\frac{I_i^{B}}{1.5d_i}\right),
\qquad
H^{B}=\sum_i I_i^{B}k_i,
$$

where the superscript \(B\) identifies the benchmark, and \(k_i\) is the holding cost per inventory unit. These equations correspond to `traditional_cost_optimization`; despite the function name, the function implements a fixed benchmark rule rather than a cost-minimization procedure.

For a constraint \(c\) with minimum service \(m_c\), penalty weight \(w_c\), and applicable locations \(L_c\), the evaluation penalty is

$$
P = \sum_c \sum_{i\in L_c} w_c\max(0,m_c-s_i).
$$

The priority-adjusted rule's holding-cost estimate and combined evaluation score are

$$
H = \sum_i I_i k_i,
\qquad
J = H + P,
$$

where \(k_i\) is the holding cost per inventory unit. The code reports \(H\) as holding cost, \(P\) as the equity penalty, and \(J\) in `EthicalSolution.total_cost`. The penalty is an evaluation term; it does not feed back into the inventory rule. Compliance is assessed separately by checking whether each applicable service level meets its minimum.

For stakeholder group \(g\), the utility score is based on mean group service \(\bar{s}_g\), priority \(q_g\), and vulnerability \(u_g\):

$$
U_g = \min\!\left(1,\bar{s}_g\left[1+(q_g-0.5)\right]+0.2u_g\right).
$$

The equity score is one minus ten times the population variance of the represented groups' mean service, bounded below by zero:

$$
E = \max\!\left(0,1-10\operatorname{Var}_g(\bar{s}_g)\right).
$$

These summary scores have deliberately limited interpretations: utility is a constructed score, and a high equity score can coexist with inadequate service for every group.

### Solution sequence

1. Define stakeholder groups, ethical principles, and equity policies.
2. Generate synthetic demand and holding-cost inputs for the listed locations.
3. Apply the priority- and vulnerability-adjusted inventory rule and estimate service.
4. Calculate holding cost, equity penalties, stakeholder utility, equity score, and compliance.
5. Evaluate the fixed cost-only benchmark rule using the same reporting measures.
6. Visualize the resulting comparisons and produce an auditable summary.

### Interpreting the results

Read the cost, service, equity, utility, and compliance measures together. The benchmark reports holding cost only; the priority-adjusted framework reports holding cost plus the penalty score, so those combined totals are not a like-for-like comparison of operating costs. The holding-cost comparison is the direct comparison of the modeled inventory carrying costs. A service threshold used in a chart is a descriptive threshold, not necessarily one of the policy constraints.

The example can illustrate how explicit priorities affect a simplified inventory rule and how policy shortfalls can be made visible. It cannot establish a real-world cost increase, a causal social benefit, regulatory compliance, or fairness across demographic groups without appropriate data, validated service models, stakeholder governance, and a decision process that actually enforces the intended policies.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import random

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class StakeholderGroup:
    """Describe a stakeholder group and its modeled service context."""

    group_id: str
    name: str
    group_type: str
    priority: float
    locations: List[str]
    service_requirements: Dict[str, float]
    population_served: int
    vulnerability_score: float


@dataclass
class EthicalPrinciple:
    """Record a principle and how it is represented in the example."""

    principle_id: str
    name: str
    description: str
    weight: float
    constraint_type: str
    implementation: str


@dataclass
class EquityConstraint:
    """Define a minimum service policy and its evaluation weight."""

    constraint_id: str
    name: str
    stakeholder_group: str
    sku_category: str
    minimum_service_level: float
    penalty_weight: float
    enforcement_type: str


@dataclass
class EthicalSolution:
    """Collect the inventory rule's decisions and evaluation measures."""

    solution_id: str
    inventory_levels: Dict[str, float]
    total_cost: float
    service_levels: Dict[str, float]
    equity_score: float
    ethical_compliance: Dict[str, bool]
    stakeholder_utility: Dict[str, float]
    transparency_report: Dict[str, Any]

In [ ]:
class EthicalOptimizationEngine:
    """Apply a priority-adjusted inventory rule and evaluate equity measures."""

    def __init__(self, network_config: Dict[str, Any]):
        self.network_config = network_config
        self.stakeholder_groups: Dict[str, StakeholderGroup] = {}
        self.ethical_principles: Dict[str, EthicalPrinciple] = {}
        self.equity_constraints: Dict[str, EquityConstraint] = {}
        self.optimization_history: List[EthicalSolution] = []
        self._initialize_ethical_framework()

    def _initialize_ethical_framework(self) -> None:
        """Register the groups, principles, and policies used in this example."""
        self.stakeholder_groups = {
            "urban_wealthy": StakeholderGroup(
                group_id="urban_wealthy",
                name="Urban High-Income Areas",
                group_type="economic",
                priority=0.6,
                locations=["DC_Urban_North", "DC_Urban_South"],
                service_requirements={"essential": 0.95, "standard": 0.90},
                population_served=500000,
                vulnerability_score=0.2,
            ),
            "urban_middle": StakeholderGroup(
                group_id="urban_middle",
                name="Urban Middle-Income Areas",
                group_type="economic",
                priority=0.8,
                locations=["DC_Urban_Central", "DC_Urban_East"],
                service_requirements={"essential": 0.97, "standard": 0.92},
                population_served=800000,
                vulnerability_score=0.4,
            ),
            "rural_general": StakeholderGroup(
                group_id="rural_general",
                name="Rural Communities",
                group_type="geographic",
                priority=0.9,
                locations=["DC_Rural_West", "DC_Rural_East"],
                service_requirements={"essential": 0.98, "standard": 0.93},
                population_served=200000,
                vulnerability_score=0.7,
            ),
            "vulnerable_elderly": StakeholderGroup(
                group_id="vulnerable_elderly",
                name="Elderly Care Facilities",
                group_type="vulnerable",
                priority=1.0,
                locations=["DC_Healthcare_North", "DC_Healthcare_South"],
                service_requirements={"essential": 0.99, "standard": 0.95},
                population_served=50000,
                vulnerability_score=0.9,
            ),
            "vulnerable_children": StakeholderGroup(
                group_id="vulnerable_children",
                name="Schools and Pediatric Care",
                group_type="vulnerable",
                priority=1.0,
                locations=["DC_Education_Central", "DC_Education_West"],
                service_requirements={"essential": 0.99, "standard": 0.95},
                population_served=100000,
                vulnerability_score=0.95,
            ),
        }

        self.ethical_principles = {
            "equity": EthicalPrinciple(
                principle_id="equity",
                name="Service Level Equity",
                description="Assess service differences across stakeholder groups.",
                weight=0.3,
                constraint_type="evaluation",
                implementation="service_shortfall_penalties_and_equity_score",
            ),
            "priority_vulnerable": EthicalPrinciple(
                principle_id="priority_vulnerable",
                name="Priority for Vulnerable Populations",
                description="Increase inventory targets for high-priority and vulnerable groups.",
                weight=0.25,
                constraint_type="heuristic_rule",
                implementation="priority_and_vulnerability_multipliers",
            ),
            "transparency": EthicalPrinciple(
                principle_id="transparency",
                name="Decision Transparency",
                description="Report inputs, evaluation measures, and decision rationale.",
                weight=0.2,
                constraint_type="reporting",
                implementation="transparency_report",
            ),
            "sustainability": EthicalPrinciple(
                principle_id="sustainability",
                name="Long-term Sustainability",
                description="Record sustainability as a governance consideration.",
                weight=0.15,
                constraint_type="reporting",
                implementation="not_quantified_in_this_example",
            ),
            "accountability": EthicalPrinciple(
                principle_id="accountability",
                name="Stakeholder Accountability",
                description="Make stakeholder measures available for review.",
                weight=0.1,
                constraint_type="reporting",
                implementation="stakeholder_impact_summary",
            ),
        }

        self.equity_constraints = {
            "essential_medicine": EquityConstraint(
                constraint_id="essential_medicine",
                name="Essential Medicine Service Level",
                stakeholder_group="all",
                sku_category="essential",
                minimum_service_level=0.99,
                penalty_weight=1000.0,
                enforcement_type="penalty_function",
            ),
            "baby_formula": EquityConstraint(
                constraint_id="baby_formula",
                name="Baby Formula Availability",
                stakeholder_group="all",
                sku_category="essential",
                minimum_service_level=0.99,
                penalty_weight=1000.0,
                enforcement_type="penalty_function",
            ),
            "basic_food": EquityConstraint(
                constraint_id="basic_food",
                name="Basic Food Staples",
                stakeholder_group="rural_general",
                sku_category="essential",
                minimum_service_level=0.95,
                penalty_weight=500.0,
                enforcement_type="penalty_function",
            ),
            "vulnerable_priority": EquityConstraint(
                constraint_id="vulnerable_priority",
                name="Vulnerable Group Priority",
                stakeholder_group="vulnerable",
                sku_category="essential",
                minimum_service_level=0.99,
                penalty_weight=750.0,
                enforcement_type="penalty_function",
            ),
        }

        configured_locations = set(self.network_config["locations"])
        assigned_locations = [
            location
            for group in self.stakeholder_groups.values()
            for location in group.locations
        ]
        if len(assigned_locations) != len(set(assigned_locations)):
            raise ValueError("Each configured location must belong to one stakeholder group.")
        if configured_locations != set(assigned_locations):
            raise ValueError("Stakeholder locations must match the configured network locations.")

    def _locations_for_constraint(self, constraint: EquityConstraint) -> List[str]:
        """Resolve a policy's stakeholder label to its applicable locations."""
        if constraint.stakeholder_group == "all":
            return list(self.network_config["locations"])
        if constraint.stakeholder_group == "vulnerable":
            return [
                location
                for group in self.stakeholder_groups.values()
                if group.group_type == "vulnerable"
                for location in group.locations
            ]
        if constraint.stakeholder_group not in self.stakeholder_groups:
            raise ValueError(
                f"Unknown stakeholder group: {constraint.stakeholder_group}"
            )
        return self.stakeholder_groups[constraint.stakeholder_group].locations

    def calculate_equity_penalty(
        self,
        service_levels: Dict[str, float],
        inventory_levels: Dict[str, float],
    ) -> float:
        """Sum weighted service shortfalls over each policy's locations."""
        del inventory_levels  # The penalty is defined on service, not inventory directly.
        total_penalty = 0.0

        for constraint in self.equity_constraints.values():
            for location in self._locations_for_constraint(constraint):
                if location not in service_levels:
                    continue
                shortfall = max(
                    0.0,
                    constraint.minimum_service_level - service_levels[location],
                )
                total_penalty += constraint.penalty_weight * shortfall

        return total_penalty

    def calculate_stakeholder_utility(
        self,
        service_levels: Dict[str, float],
        inventory_levels: Dict[str, float],
    ) -> Dict[str, float]:
        """Calculate a bounded utility score from group service and attributes."""
        del inventory_levels  # Utility is based on service and group attributes.
        utilities: Dict[str, float] = {}

        for group_id, group in self.stakeholder_groups.items():
            group_services = [
                service_levels[location]
                for location in group.locations
                if location in service_levels
            ]
            if not group_services:
                utilities[group_id] = 0.0
                continue

            average_service = float(np.mean(group_services))
            priority_multiplier = 1.0 + (group.priority - 0.5)
            vulnerability_bonus = group.vulnerability_score * 0.2
            utilities[group_id] = min(
                1.0,
                average_service * priority_multiplier + vulnerability_bonus,
            )

        return utilities

    def optimize_ethical_inventory(
        self,
        base_demand: Dict[str, float],
        base_costs: Dict[str, float],
    ) -> EthicalSolution:
        """Apply the inventory heuristic and calculate its evaluation measures."""
        locations = list(self.network_config["locations"])
        if set(base_demand) != set(locations) or set(base_costs) != set(locations):
            raise ValueError("Demand and cost inputs must cover exactly the configured locations.")
        if any(base_demand[location] <= 0 for location in locations):
            raise ValueError("Demand must be positive at every configured location.")
        if any(base_costs[location] < 0 for location in locations):
            raise ValueError("Holding costs must be non-negative.")

        optimal_inventory: Dict[str, float] = {}
        service_levels: Dict[str, float] = {}

        for location in locations:
            demand = base_demand[location]
            group_id = self._get_location_stakeholder(location)
            group = self.stakeholder_groups[group_id]

            base_stock = demand * 1.5
            if group.vulnerability_score > 0.7:
                base_stock *= 1.3
            if group.priority > 0.8:
                base_stock *= 1.2

            optimal_inventory[location] = base_stock
            service_levels[location] = min(0.99, base_stock / (demand * 2.0))

        holding_cost = sum(
            optimal_inventory[location] * base_costs[location]
            for location in locations
        )
        equity_penalty = self.calculate_equity_penalty(
            service_levels, optimal_inventory
        )
        total_cost = holding_cost + equity_penalty
        stakeholder_utility = self.calculate_stakeholder_utility(
            service_levels, optimal_inventory
        )
        equity_score = self._calculate_equity_score(service_levels)
        ethical_compliance = self._check_ethical_compliance(service_levels)
        transparency_report = self._generate_transparency_report(
            optimal_inventory,
            service_levels,
            holding_cost,
            equity_penalty,
        )

        solution = EthicalSolution(
            solution_id=f"ETH_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            inventory_levels=optimal_inventory,
            total_cost=total_cost,
            service_levels=service_levels,
            equity_score=equity_score,
            ethical_compliance=ethical_compliance,
            stakeholder_utility=stakeholder_utility,
            transparency_report=transparency_report,
        )
        self.optimization_history.append(solution)
        return solution

    def _get_location_stakeholder(self, location: str) -> str:
        """Return the unique group assigned to a configured location."""
        for group_id, group in self.stakeholder_groups.items():
            if location in group.locations:
                return group_id
        raise ValueError(f"No stakeholder group is assigned to location {location!r}.")

    def _calculate_equity_score(self, service_levels: Dict[str, float]) -> float:
        """Convert variance in group-average service into a bounded score."""
        group_services = []
        for group in self.stakeholder_groups.values():
            levels = [
                service_levels[location]
                for location in group.locations
                if location in service_levels
            ]
            if levels:
                group_services.append(float(np.mean(levels)))

        if not group_services:
            return 0.0
        variance = float(np.var(group_services))
        return max(0.0, 1.0 - variance * 10.0)

    def _check_ethical_compliance(
        self,
        service_levels: Dict[str, float],
    ) -> Dict[str, bool]:
        """Check each service minimum at every location covered by its policy."""
        compliance: Dict[str, bool] = {}

        for constraint_id, constraint in self.equity_constraints.items():
            locations = self._locations_for_constraint(constraint)
            compliance[constraint_id] = all(
                location in service_levels
                and service_levels[location] >= constraint.minimum_service_level
                for location in locations
            )

        return compliance

    def _generate_transparency_report(
        self,
        inventory: Dict[str, float],
        service: Dict[str, float],
        holding_cost: float,
        penalty: float,
    ) -> Dict[str, Any]:
        """Summarize the rule, its evaluation, and the information it uses."""
        if not service:
            raise ValueError("A transparency report requires at least one service value.")

        return {
            "optimization_method": "priority_adjusted_inventory_heuristic",
            "total_cost_breakdown": {
                "holding_cost": holding_cost,
                "equity_penalty": penalty,
                "total_cost": holding_cost + penalty,
            },
            "service_level_summary": {
                "average_service": float(np.mean(list(service.values()))),
                "min_service": float(np.min(list(service.values()))),
                "max_service": float(np.max(list(service.values()))),
                "service_variance": float(np.var(list(service.values()))),
            },
            "ethical_principles_recorded": list(self.ethical_principles.keys()),
            "stakeholder_impact": self.calculate_stakeholder_utility(service, inventory),
            "decision_rationale": self._generate_decision_rationale(),
            "audit_timestamp": datetime.now().isoformat(timespec="seconds"),
        }

    def _generate_decision_rationale(self) -> List[str]:
        """State the modeled rule and the limits of its policy evaluation."""
        return [
            "Inventory targets start at 1.5 times location demand.",
            "The rule applies fixed multipliers for high vulnerability and priority.",
            "Location service is estimated as inventory divided by twice demand, capped at 99%.",
            "Weighted service shortfalls are reported as an equity penalty; they do not alter inventory decisions.",
            "Policy labels share a location-level service proxy because SKU-level demand is not modeled.",
        ]

In [ ]:
print("Initializing the equitable service-level example...")
print("=" * 55)

network_config = {
    "locations": [
        "DC_Urban_North",
        "DC_Urban_South",
        "DC_Urban_Central",
        "DC_Urban_East",
        "DC_Rural_West",
        "DC_Rural_East",
        "DC_Healthcare_North",
        "DC_Healthcare_South",
        "DC_Education_Central",
        "DC_Education_West",
    ]
}

ethical_engine = EthicalOptimizationEngine(network_config)

print(f"Stakeholder groups: {len(ethical_engine.stakeholder_groups)}")
for group in ethical_engine.stakeholder_groups.values():
    print(
        f"  {group.name}: priority={group.priority:.1f}, "
        f"vulnerability={group.vulnerability_score:.1f}"
    )
    print(f"    Locations: {', '.join(group.locations)}")
    print(f"    Recorded service requirements: {group.service_requirements}")

print(f"\nRecorded principles: {len(ethical_engine.ethical_principles)}")
for principle in ethical_engine.ethical_principles.values():
    print(f"  {principle.name}: representation={principle.implementation}")

print(f"\nService policies: {len(ethical_engine.equity_constraints)}")
for constraint in ethical_engine.equity_constraints.values():
    print(
        f"  {constraint.name}: minimum="
        f"{constraint.minimum_service_level:.0%}, "
        f"group={constraint.stakeholder_group}, "
        f"penalty weight={constraint.penalty_weight:.0f}"
    )

In [ ]:
print("\nGenerating synthetic demand and holding-cost inputs...")
print("=" * 55)

locations = network_config["locations"]
base_demand: Dict[str, float] = {}
base_costs: Dict[str, float] = {}

for location in locations:
    if "Urban" in location:
        demand = random.uniform(80, 120)
        cost = 2.0
    elif "Rural" in location:
        demand = random.uniform(40, 80)
        cost = 1.5
    elif "Healthcare" in location:
        demand = random.uniform(30, 60)
        cost = 2.5
    else:
        demand = random.uniform(20, 50)
        cost = 2.0

    base_demand[location] = demand
    base_costs[location] = cost

print("Sample demand and holding-cost inputs:")
print("-" * 55)
for location in locations[:5]:
    print(
        f"{location:<22} demand={base_demand[location]:6.1f} units, "
        f"holding cost=${base_costs[location]:.2f} per unit"
    )
print(f"... and {len(locations) - 5} additional locations")

ethical_solution = ethical_engine.optimize_ethical_inventory(base_demand, base_costs)
holding_cost = ethical_solution.transparency_report["total_cost_breakdown"]["holding_cost"]
equity_penalty = ethical_solution.transparency_report["total_cost_breakdown"]["equity_penalty"]

assert set(ethical_solution.inventory_levels) == set(locations)
assert set(ethical_solution.service_levels) == set(locations)
assert all(0.0 <= level <= 0.99 for level in ethical_solution.service_levels.values())
assert np.isclose(ethical_solution.total_cost, holding_cost + equity_penalty)

print("\nPriority-adjusted heuristic evaluation:")
print("-" * 55)
print(f"Holding-cost estimate: ${holding_cost:.2f}")
print(f"Equity penalty:        ${equity_penalty:.2f} (dollar-equivalent score)")
print(f"Combined score:        ${ethical_solution.total_cost:.2f}")
print(f"Equity score:          {ethical_solution.equity_score:.3f}")
print(
    "Mean location service: "
    f"{np.mean(list(ethical_solution.service_levels.values())):.1%}"
)

print("\nStakeholder utility scores:")
for group_id, utility in ethical_solution.stakeholder_utility.items():
    group = ethical_engine.stakeholder_groups[group_id]
    print(f"  {group.name:<30} {utility:.3f}")

print("\nPolicy compliance checks:")
compliance_count = sum(ethical_solution.ethical_compliance.values())
total_constraints = len(ethical_solution.ethical_compliance)
print(f"Policies met: {compliance_count}/{total_constraints}")
print(f"Policy compliance rate: {compliance_count / total_constraints:.1%}")
for constraint_id, compliant in ethical_solution.ethical_compliance.items():
    constraint = ethical_engine.equity_constraints[constraint_id]
    status = "Met" if compliant else "Not met"
    print(
        f"  {status}: {constraint.name} "
        f"(minimum {constraint.minimum_service_level:.0%})"
    )

In [ ]:
def traditional_cost_optimization(
    demand: Dict[str, float],
    costs: Dict[str, float],
) -> Dict[str, Dict[str, float] | float]:
    """Construct the fixed cost-only benchmark rule used for context."""
    if set(demand) != set(costs):
        raise ValueError("Demand and cost inputs must have matching locations.")
    if any(value <= 0 for value in demand.values()):
        raise ValueError("Demand must be positive at every location.")
    if any(value < 0 for value in costs.values()):
        raise ValueError("Holding costs must be non-negative.")

    inventory: Dict[str, float] = {}
    service_levels: Dict[str, float] = {}
    for location, location_demand in demand.items():
        inventory[location] = location_demand * 1.2
        service_levels[location] = min(
            0.90,
            inventory[location] / (location_demand * 1.5),
        )

    total_holding_cost = sum(
        inventory[location] * costs[location]
        for location in demand
    )
    return {
        "inventory": inventory,
        "service_levels": service_levels,
        "holding_cost": total_holding_cost,
    }


traditional_result = traditional_cost_optimization(base_demand, base_costs)
traditional_equity_penalty = ethical_engine.calculate_equity_penalty(
    traditional_result["service_levels"], traditional_result["inventory"]
)
traditional_equity_score = ethical_engine._calculate_equity_score(
    traditional_result["service_levels"]
)
traditional_utility = ethical_engine.calculate_stakeholder_utility(
    traditional_result["service_levels"], traditional_result["inventory"]
)

ethical_holding_cost = ethical_solution.transparency_report["total_cost_breakdown"]["holding_cost"]
traditional_holding_cost = traditional_result["holding_cost"]
holding_cost_change = (
    (ethical_holding_cost - traditional_holding_cost)
    / traditional_holding_cost
    * 100.0
    if not np.isclose(traditional_holding_cost, 0.0)
    else float("nan")
)

print("\nComparison of the heuristic and the fixed cost-only benchmark rule")
print("=" * 65)
print("Holding-cost estimates:")
print(f"  Cost-only benchmark:       ${traditional_holding_cost:.2f}")
print(f"  Priority-adjusted rule:    ${ethical_holding_cost:.2f}")
print(f"  Holding-cost change:       {holding_cost_change:.1f}%")
print(
    "The priority-adjusted evaluation also reports an equity penalty of "
    f"${equity_penalty:.2f}; the benchmark's corresponding penalty is "
    f"${traditional_equity_penalty:.2f}. These penalties are evaluation "
    "scores, not measured operating expenses."
)

print("\nEquity score:")
print(f"  Cost-only benchmark:       {traditional_equity_score:.3f}")
print(f"  Priority-adjusted rule:    {ethical_solution.equity_score:.3f}")
if not np.isclose(traditional_equity_score, 0.0):
    equity_change = (
        (ethical_solution.equity_score - traditional_equity_score)
        / traditional_equity_score
        * 100.0
    )
    print(f"  Relative score change:     {equity_change:.1f}%")
else:
    print("  Relative score change:     undefined because the benchmark score is zero")

traditional_average_service = float(
    np.mean(list(traditional_result["service_levels"].values()))
)
ethical_average_service = float(
    np.mean(list(ethical_solution.service_levels.values()))
)
print("\nMean location service:")
print(f"  Cost-only benchmark:       {traditional_average_service:.1%}")
print(f"  Priority-adjusted rule:    {ethical_average_service:.1%}")

print("\nStakeholder utility scores:")
print(f"{'Stakeholder group':<32} {'Benchmark':>11} {'Heuristic':>11}")
print("-" * 56)
for group_id, group in ethical_engine.stakeholder_groups.items():
    print(
        f"{group.name:<32} "
        f"{traditional_utility[group_id]:>11.3f} "
        f"{ethical_solution.stakeholder_utility[group_id]:>11.3f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    "Priority-Adjusted Inventory Rule and Cost-Only Benchmark",
    fontsize=16,
    fontweight="bold",
)

methods = ["Cost-only benchmark", "Priority-adjusted rule"]

# Holding costs are compared separately from the equity penalty score.
ax1 = axes[0, 0]
holding_costs = [traditional_holding_cost, ethical_holding_cost]
bars = ax1.bar(methods, holding_costs, color=["#FF6B6B", "#4ECDC4"], alpha=0.8)
ax1.set_ylabel("Holding cost (dollars)")
ax1.set_title("Holding-Cost Estimates", fontweight="bold")
ax1.tick_params(axis="x", rotation=15)
ax1.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, holding_costs):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${value:.0f}",
        ha="center",
        va="bottom",
    )

ax2 = axes[0, 1]
equity_scores = [traditional_equity_score, ethical_solution.equity_score]
bars = ax2.bar(methods, equity_scores, color=["#FF6B6B", "#4ECDC4"], alpha=0.8)
ax2.set_ylabel("Equity score")
ax2.set_title("Variance-Based Equity Score", fontweight="bold")
ax2.set_ylim(0, 1.05)
ax2.tick_params(axis="x", rotation=15)
ax2.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, equity_scores):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

ax3 = axes[0, 2]
traditional_services = list(traditional_result["service_levels"].values())
ethical_services = list(ethical_solution.service_levels.values())
service_bins = np.linspace(0.0, 1.0, 11)
ax3.hist(
    traditional_services,
    bins=service_bins,
    alpha=0.55,
    label="Cost-only benchmark",
    color="#FF6B6B",
    density=True,
)
ax3.hist(
    ethical_services,
    bins=service_bins,
    alpha=0.55,
    label="Priority-adjusted rule",
    color="#4ECDC4",
    density=True,
)
ax3.set_xlabel("Location service level")
ax3.set_ylabel("Density")
ax3.set_title("Service-Level Distributions", fontweight="bold")
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
group_names = [
    group.name for group in ethical_engine.stakeholder_groups.values()
]
group_ids = list(ethical_engine.stakeholder_groups)
benchmark_utilities = [traditional_utility[group_id] for group_id in group_ids]
priority_utilities = [
    ethical_solution.stakeholder_utility[group_id] for group_id in group_ids
]
positions = np.arange(len(group_names))
bar_width = 0.35
ax4.bar(
    positions - bar_width / 2,
    benchmark_utilities,
    bar_width,
    label="Cost-only benchmark",
    color="#FF6B6B",
    alpha=0.8,
)
ax4.bar(
    positions + bar_width / 2,
    priority_utilities,
    bar_width,
    label="Priority-adjusted rule",
    color="#4ECDC4",
    alpha=0.8,
)
ax4.set_xlabel("Stakeholder group")
ax4.set_ylabel("Constructed utility score")
ax4.set_title("Stakeholder Utility Scores", fontweight="bold")
ax4.set_xticks(positions)
ax4.set_xticklabels(group_names, rotation=45, ha="right")
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)

ax5 = axes[1, 1]
threshold = 0.95
benchmark_count = sum(level >= threshold for level in traditional_services)
priority_count = sum(level >= threshold for level in ethical_services)
location_count = len(traditional_services)
threshold_shares = [
    benchmark_count / location_count,
    priority_count / location_count,
]
bars = ax5.bar(methods, threshold_shares, color=["#FF6B6B", "#4ECDC4"], alpha=0.8)
ax5.set_ylabel("Share of locations at or above 95% service")
ax5.set_title("Descriptive 95% Service Threshold", fontweight="bold")
ax5.set_ylim(0, 1.05)
ax5.tick_params(axis="x", rotation=15)
ax5.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, threshold_shares):
    ax5.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.1%}",
        ha="center",
        va="bottom",
    )

# The figure shows evaluated endpoints; it does not fabricate a weight sweep.
ax6 = axes[1, 2]
ax6.scatter(
    traditional_holding_cost,
    traditional_equity_score,
    color="#FF6B6B",
    s=100,
    label="Cost-only benchmark",
    zorder=3,
)
ax6.scatter(
    ethical_holding_cost,
    ethical_solution.equity_score,
    color="#4ECDC4",
    s=100,
    label="Priority-adjusted rule",
    zorder=3,
)
ax6.set_xlabel("Holding cost (dollars; excludes penalty)")
ax6.set_ylabel("Variance-based equity score")
ax6.set_title("Evaluated Cost and Equity Points", fontweight="bold")
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("\nTRANSPARENCY AND ACCOUNTABILITY SUMMARY")
print("=" * 60)

transparency = ethical_solution.transparency_report
print("Evaluation rule:")
print(f"  {transparency['optimization_method']}")
print(f"  Report timestamp: {transparency['audit_timestamp']}")
print(
    "  Principles recorded: "
    + ", ".join(transparency["ethical_principles_recorded"])
)

cost_breakdown = transparency["total_cost_breakdown"]
print("\nEvaluation score breakdown:")
print(f"  Holding-cost estimate: ${cost_breakdown['holding_cost']:.2f}")
print(f"  Equity penalty score:  ${cost_breakdown['equity_penalty']:.2f}")
print(f"  Combined score:        ${cost_breakdown['total_cost']:.2f}")
if not np.isclose(cost_breakdown["total_cost"], 0.0):
    penalty_share = (
        cost_breakdown["equity_penalty"]
        / cost_breakdown["total_cost"]
        * 100.0
    )
    print(f"  Penalty share of combined score: {penalty_share:.1f}%")
else:
    print("  Penalty share of combined score: undefined because the score is zero")

service_summary = transparency["service_level_summary"]
print("\nLocation service summary:")
print(f"  Mean:     {service_summary['average_service']:.1%}")
print(f"  Minimum:  {service_summary['min_service']:.1%}")
print(f"  Maximum:  {service_summary['max_service']:.1%}")
print(f"  Variance: {service_summary['service_variance']:.4f}")

print("\nDecision rationale:")
for number, rationale in enumerate(transparency["decision_rationale"], start=1):
    print(f"  {number}. {rationale}")

print("\nStakeholder impact measures:")
for group_id, utility in transparency["stakeholder_impact"].items():
    group = ethical_engine.stakeholder_groups[group_id]
    print(f"  {group.name}")
    print(f"    Constructed utility score: {utility:.3f}")
    print(f"    Priority: {group.priority:.1f}")
    print(f"    Vulnerability: {group.vulnerability_score:.1f}")
    print(f"    Population represented: {group.population_served:,}")

print("\nPolicy evaluation:")
print(f"  Policies met: {compliance_count}/{total_constraints}")
print(f"  Compliance rate: {compliance_count / total_constraints:.1%}")
print("  A passing check applies only to the modeled location-level proxy.")

print("\nInterpretation limits:")
print("  The combined score is not a measured operating expense or an optimal objective value.")
print("  Utility and equity scores are descriptive measures defined by this example.")
print("  Results do not establish legal compliance, causal social impact, or fairness.")

## Interpreting the Framework

### What the comparison can show

The benchmark and priority-adjusted rule apply different fixed inventory rules to the same synthetic demand and holding-cost inputs. Their holding-cost estimates, service proxies, stakeholder utility scores, and variance-based equity scores describe the outcomes of those rules for this scenario. The equity penalty makes modeled service shortfalls visible, while compliance checks state whether every location covered by a policy reaches its specified threshold.

A larger equity score means that the represented groups have more similar average service under the score's variance-based definition. It does not mean that service is adequate: similarly low service across groups can still yield a high score. Likewise, a higher utility score is not a direct measure of welfare; it follows the specified service, priority, and vulnerability formula.

### Trade-offs and governance

Inventory multipliers can raise holding cost while changing service and the distribution of service across groups. The direction and magnitude depend on the input data and the rule. The penalty weights also affect the reported combined evaluation score, but they do not change inventory decisions in this implementation. For this reason, the notebook does not demonstrate an optimized cost-equity trade-off curve or establish a typical percentage increase in cost.

A real equitable inventory program would require suitably disaggregated demand and service data, explicit decisions about which populations and products each policy covers, stakeholder review of targets and weights, and a decision mechanism that can enforce the intended service commitments. Monitoring should distinguish measured outcomes from modeled proxies and should document assumptions, uncertainty, and accountability.